# Plan Visualization

LangGOAP ships built-in renderers that turn a `Plan` into Mermaid, Graphviz DOT,
and plain ASCII representations. Every renderer is pure-Python and dependency-free:
the Mermaid output renders inline in Jupyter; DOT can be piped to Graphviz; ASCII is
perfect for logs and terminals.

When a plan carries CSP metadata (from the A* → CSP pipeline), the renderers
additionally show parallel clusters, resource usage, schedule Gantt charts and
constraint satisfaction indicators.

## Setup

Define a three-step linear pipeline and plan a goal against it.

In [ ]:
from datetime import timedelta

from langgoap import (
    ActionSpec,
    ConstraintSpec,
    GoalSpec,
    PlanningState,
)
from langgoap.planner.pipeline import plan as pipeline_plan

actions = [
    ActionSpec(
        name="collect_specs",
        effects={"specs_collected": True},
        resources={"tokens": 100.0, "cost_usd": 0.10},
    ),
    ActionSpec(
        name="design_layout",
        preconditions={"specs_collected": True},
        effects={"layout_designed": True},
        resources={"tokens": 200.0, "cost_usd": 0.20},
    ),
    ActionSpec(
        name="compile_report",
        preconditions={"layout_designed": True},
        effects={"report_ready": True},
        resources={"tokens": 150.0, "cost_usd": 0.15},
    ),
]

start = PlanningState.from_dict({})
goal = GoalSpec(
    conditions={"report_ready": True},
    constraints=(
        ConstraintSpec(key="tokens", max=1000.0),
        ConstraintSpec(key="cost_usd", max=1.0),
    ),
)

plan = pipeline_plan(start, goal, actions)
print(plan)

## Mermaid flowchart

`Plan.visualize()` auto-detects IPython and returns an `IPython.display.Markdown`
that Jupyter renders as a Mermaid diagram.  Dependency arrows come from the
same effect→precondition graph the scheduler uses.

In [ ]:
plan.visualize(format="mermaid")

## ASCII tree

For logs and terminals, `Plan.to_ascii()` returns a plain-text tree with
per-action dependency indices and a resource usage summary.

In [ ]:
print(plan.to_ascii())

## Graphviz DOT

`Plan.to_dot()` returns Graphviz source code. Pipe it to the `dot` binary
for high-quality renders, or pass it to the `graphviz` Python package.

In [ ]:
print(plan.to_dot())

## Scheduled plan with parallelism

Adding `duration` to actions triggers the CSP temporal scheduler.
Actions with no mutual dependencies are scheduled in parallel —
renderers group them visually.

In [ ]:
parallel_actions = [
    ActionSpec(
        name="fetch_source_a",
        effects={"source_a": True},
        duration=timedelta(seconds=2),
        resources={"cost_usd": 0.10},
    ),
    ActionSpec(
        name="fetch_source_b",
        effects={"source_b": True},
        duration=timedelta(seconds=3),
        resources={"cost_usd": 0.15},
    ),
    ActionSpec(
        name="merge_sources",
        preconditions={"source_a": True, "source_b": True},
        effects={"merged": True},
        duration=timedelta(seconds=1),
        resources={"cost_usd": 0.05},
    ),
]

scheduled_plan = pipeline_plan(
    PlanningState.from_dict({}),
    GoalSpec(
        conditions={"merged": True},
        constraints=(ConstraintSpec(key="cost_usd", max=1.0),),
    ),
    parallel_actions,
)
assert scheduled_plan is not None
print(f"Makespan: {scheduled_plan.metadata.csp.makespan}")
scheduled_plan.visualize(format="mermaid")

## Gantt chart

`format="gantt"` renders a Mermaid gantt from the CSP schedule.
`format="ascii_gantt"` gives the same visualization as plain text.

In [ ]:
scheduled_plan.visualize(format="gantt")

In [ ]:
print(scheduled_plan.visualize(format="ascii_gantt"))

## Constraint violation reporting

When a plan violates its budget, renderers mark the violated resource
as `VIOLATED` so it stands out in the diagram.

In [ ]:
tight_goal = GoalSpec(
    conditions={"report_ready": True},
    constraints=(ConstraintSpec(key="tokens", max=100.0),),  # tight
)
tight_plan = pipeline_plan(start, tight_goal, actions)
print(tight_plan.to_ascii())

## Saving to disk

`Plan.save()` writes a rendered representation to a file. The format
is inferred from the extension: `.mmd` → Mermaid, `.dot` → DOT,
anything else → ASCII.  Pass `format=` to override.

In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
plan.save(tmp / "plan.mmd")
plan.save(tmp / "plan.dot")
plan.save(tmp / "plan.txt")
print(sorted(p.name for p in tmp.iterdir()))